# 01 — HFBTP Dataset Audit

**Purpose:** verify file identity (checksums), schema, data quality, and grid coverage of the raw HFBTP benchmark against the audit references recorded in `docs/hfbtp_audit_report.md` and `data/manifests/hfbtp_manifest.yaml`. The loader never mutates the raw CSV; everything below is recomputed from the actual file.

This notebook separates **verification** (recomputed here) from **documented assumptions** (provenance claims inherited from the planning workspace).

## Setup

In [1]:
from pathlib import Path
import sys, json

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from fabric_perf.data.hfbtp import HFBTPDatasetLoader
from fabric_perf.utils.helpers import results_dir, figures_dir, save_json, utc_timestamp

loader = HFBTPDatasetLoader(ROOT / "data" / "raw" / "hfbtp")
print("Raw file:", loader._find_raw_file())

Raw file: D:\Reseach_Model\AI-Assisted-Permissioned-Blockchain-Architecture-\data\raw\hfbtp\HFBTP.csv


## 1. File identity — checksums recomputed from the actual file

In [2]:
checksums = loader.compute_checksums()
checksums
refs = {
    "md5_ref": "E9275C49A2D7DC1F2A4015B6D8B28482",
    "sha256_ref": "CE295D6451787FEFCFE2101579238CDA4BEDE8539BC302E6A482AA4E5782BD5E",
}
identity_ok = (checksums["md5"] == refs["md5_ref"]) and (checksums["sha256"] == refs["sha256_ref"])
print("Checksum identity vs audit record:", identity_ok)
assert identity_ok, "Raw file changed/corrupted - audit reference mismatch!"


Checksum identity vs audit record: True


## 2. Schema and data quality (strict validation)

In [3]:
df = loader.load_raw()
report = loader.validate(df)
print(f"rows={report['row_count']}, cols={report['col_count']}")
print("schema_valid:", report['schema_valid'])
print("missing values:", int(report['has_nulls']), "| total nulls:", sum(report['missing_values'].values()))
print("duplicate rows:", report['duplicate_rows'])
import pandas as pd
display(pd.DataFrame({'dtype': report['dtypes']}))
assert report['schema_valid'] and not report['has_nulls'] and report['duplicate_rows'] == 0

rows=24687, cols=8
schema_valid: True
missing values: 0 | total nulls: 0
duplicate rows: 0


,dtype
Set Transaction Arrival Rate,float64
Actual Transaction Arrival Rate,float64
Block Size,float64
Orderers,int64
Throughput,float64
Avg Latency,float64
Min Latency,float64
Max Latency,float64


## 3. Configuration grid structure and coverage

In [4]:
grid = loader.describe_grid(df)
grid
cov_ok = (grid['missing_cells'] == 117) and (grid['observed_cells'] == 24687) and (grid['max_rows_per_cell'] == 1)
print("Grid audit references verified (117 absent / one obs per cell):", cov_ok)
assert cov_ok

Grid audit references verified (117 absent / one obs per cell): True


## 4. Which cells are missing? (descriptive, no imputation)

In [5]:
import itertools
present = set(map(tuple, df[['Set Transaction Arrival Rate','Block Size','Orderers']].itertuples(index=False)))
set_vals = sorted(df['Set Transaction Arrival Rate'].unique())
block_vals = sorted(df['Block Size'].unique())
ord_vals = sorted(df['Orderers'].unique())
absent = [k for k in itertools.product(set_vals, block_vals, ord_vals) if k not in present]
import pandas as pd
absent_df = pd.DataFrame(absent, columns=['set','block','orderers'])
print("absent cells:", len(absent_df))
display(absent_df.groupby('orderers').size())
display(absent_df.groupby('set').size().sort_values(ascending=False).head(10))

absent cells: 117


orderers
5    39
9    78
dtype: int64

set
10.0    3
15.0    3
20.0    3
25.0    3
30.0    3
35.0    3
40.0    3
45.0    3
50.0    3
55.0    3
dtype: int64

## 5. Target ranges and the set-vs-actual rate divergence

In [6]:
ranges = {c: {'min': df[c].min(), 'max': df[c].max(), 'mean': round(df[c].mean(),3), 'median': df[c].median()} for c in ['Throughput','Avg Latency','Min Latency','Max Latency']}
import pandas as pd
display(pd.DataFrame(ranges).T)
div = (df['Set Transaction Arrival Rate'] - df['Actual Transaction Arrival Rate']).abs()
print("rows diverging >5 tx/s:", int((div>5).sum()), "| >1 tx/s:", int((div>1).sum()))
print("(audit report expected: 20 rows >5, concentrated at set rates 185-200 — saturation artifact, not an error)")

,min,max,mean,median
Throughput,9.80,197.30,91.712,95.60
Avg Latency,0.08,2.67,0.895,0.85
Min Latency,0.01,0.58,0.166,0.16
Max Latency,0.15,5.14,1.915,2.13


rows diverging >5 tx/s: 20 | >1 tx/s: 427
(audit report expected: 20 rows >5, concentrated at set rates 185-200 — saturation artifact, not an error)


## 6. Persist the audit report

In [7]:
audit = loader.audit()
audit['audit_reference_checks'] = {
    'md5_matches_record': checksums['md5'] == refs['md5_ref'],
    'sha256_matches_record': checksums['sha256'] == refs['sha256_ref'],
    'grid_coverage_matches_record': cov_ok,
}
audit['executed_at_utc'] = utc_timestamp()
out = results_dir() / "hfbtp_audit_report.json"
save_json(audit, out)
print("saved:", out)

saved: D:\Reseach_Model\AI-Assisted-Permissioned-Blockchain-Architecture-\experiments\results\hfbtp_audit_report.json


## What this audit does NOT establish

- **Provenance claims** (paper, tool versions, hardware) are inherited from the LearningChain paper text in the planning workspace — verified there, not re-derivable from the CSV.
- **Why 117 cells are absent** is unknown; treated as design gaps, never imputed.
- **One observation per cell** means no noise floor: repeatability, uncertainty, and any variability claim are out of scope for this dataset (see `docs/methodology.md`).